In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup, Comment
import time
import re
import numpy as np

In [5]:
BASE_URL = "https://www.sports-reference.com"
START_URL = f"{BASE_URL}/cbb/seasons/men/2026-school-stats.html"

def get_team_links():
    print("Fetching school list...")
    # It's good practice to add a User-Agent so the site doesn't think you're a bot immediately
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(START_URL, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    
    table = soup.find("table", {"id": "basic_school_stats"})
    links = []
    
    for row in table.find("tbody").find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue
        school_cell = row.find("td", {"data-stat": "school_name"})
        if school_cell and school_cell.find("a"):
            links.append({
                "school": school_cell.text,
                "url": BASE_URL + school_cell.find("a")["href"]
            })
    return links

# Use a real User-Agent to avoid getting flagged
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

def scrape_advanced_table(url):
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # 1. Look for the table in the normal HTML first
    table = soup.find('table', id='advanced')
    
    # 2. If not found, search inside the HTML comments
    if not table:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if 'id="advanced"' in comment:
                # Create a mini-soup from the comment string
                comment_soup = BeautifulSoup(comment, 'html.parser')
                table = comment_soup.find('table', id='advanced')
                break
                
    if table:
        # read_html returns a list of dataframes; we want the first one
        df = pd.read_html(str(table))[0]
        
        # Clean up the double-header (MultiIndex)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(0)
            
        return df
    return None

# Example usage for Abilene Christian
acu_url = "https://www.sports-reference.com/cbb/schools/abilene-christian/men/2026.html"
df = scrape_advanced_table(acu_url)

if df is not None:
    print("Successfully captured Advanced Table!")
    print(df.head())
else:
    print("Still couldn't find the table.")

Still couldn't find the table.


In [6]:
df_bio = pd.read_csv('player_bio_2026.csv')
df_bio.head()

,Player,#,Class,Pos,Height,Weight,Hometown,High School,RSCI Top 100,Summary,School
0,Bradyn Hubbard,15,SR,F,6-7,225.0,"Tulsa, OK",Tulsa Memorial (OK),NaN,"16.1 Pts, 4.8 Reb, 1.8 Ast",Abilene Christian
1,Rich Smith,4,SR,G,6-4,175.0,"Bronx, NY",Monsignor Scanlan (NY),NaN,"9.6 Pts, 3.9 Reb, 4.7 Ast",Abilene Christian
2,Chilaydrien Newton,5,SO,G,6-3,175.0,"Simsboro, LA",Simsboro (LA); Wasatch Academy (UT),NaN,"9.4 Pts, 2.2 Reb, 0.9 Ast",Abilene Christian
3,Zy Wright,0,SR,G,6-5,NaN,"Lincolnton, GA",Grovetown (GA); Aquinas (GA),NaN,"6.2 Pts, 2.4 Reb, 0.7 Ast",Abilene Christian
4,Yaniel Rivera,2,JR,G,6-4,175.0,"Bayamon, Puerto Rico",Dade Christian (FL),NaN,"7.0 Pts, 1.5 Reb, 2.5 Ast",Abilene Christian


In [7]:
df_stats = pd.read_csv('player_stats_2026.csv')
df_stats

,Rk,Player,Pos,G,GS,MP,FG,FGA,FG%,3P,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,School
0,1.0,Bradyn Hubbard,F,26,23,27.9,5.3,13.0,0.409,1.4,...,3.7,4.8,1.8,1.9,0.2,2.3,2.8,16.1,NaN,Abilene Christian
1,2.0,Rich Smith,G,25,22,29.9,3.4,7.0,0.483,0.1,...,2.2,3.9,4.7,1.8,0.4,2.8,2.8,9.6,NaN,Abilene Christian
2,3.0,Chilaydrien Newton,G,23,17,24.5,3.4,7.7,0.449,1.1,...,1.5,2.2,0.9,1.2,0.1,1.3,2.3,9.4,NaN,Abilene Christian
3,4.0,Yaniel Rivera,G,22,15,24.3,2.2,6.3,0.345,1.5,...,1.3,1.5,2.5,1.2,0.2,1.5,1.1,7.0,NaN,Abilene Christian
4,5.0,Zy Wright,G,26,4,16.8,2.1,4.8,0.435,0.7,...,1.4,2.4,0.7,0.5,0.2,0.9,1.4,6.2,NaN,Abilene Christian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5367,10.0,Andrew King,G,28,1,18.0,0.8,1.8,0.469,0.2,...,2.4,3.1,3.0,0.5,0.0,1.3,1.4,2.9,NaN,Youngstown State
5368,11.0,Tyler Robinett,F,12,4,11.5,1.2,2.8,0.412,0.2,...,1.2,2.5,0.4,0.2,0.5,0.4,1.7,2.8,NaN,Youngstown State
5369,12.0,Derrick Anderson,G,6,0,5.8,1.2,2.2,0.538,0.2,...,0.5,0.7,1.0,0.8,0.2,0.0,0.2,2.7,NaN,Youngstown State
5370,13.0,Shaheed Solebo,G,7,0,6.4,0.6,1.6,0.364,0.1,...,0.9,1.3,0.9,0.0,0.0,0.7,0.6,2.3,NaN,Youngstown State


In [8]:
df = pd.merge(df_bio, df_stats, on = "Player", how = "outer")
df.head()

,Player,#,Class,Pos_x,Height,Weight,Hometown,High School,RSCI Top 100,Summary,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,School_y
0,A'Meir Williams,27.0,FR,F,6-8,230.0,"Kennedale, TX",Kennedale (TX),NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A'lahn Sumler,1.0,SR,G,6-3,185.0,"Buford, GA",Buford (GA),NaN,"18.6 Pts, 4.3 Reb, 3.6 Ast",...,3.3,4.3,3.6,0.8,0.2,2.2,2.0,18.6,NaN,Charleston Southern
2,A.J. Gladieux,10.0,FR,G,6-2,170.0,"Holly Springs, NC",Apex Friendship (NC); Grace Christian (NC),NaN,"1.5 Pts, 0.8 Reb, 0.8 Ast",...,0.5,0.8,0.8,0.0,0.0,0.5,0.0,1.5,NaN,Charleston Southern
3,A.J. Lopez,6.0,SR,G,6-5,170.0,"Queens, NY",Mount Zion Prep,NaN,"12.9 Pts, 2.3 Reb, 1.2 Ast",...,1.9,2.3,1.2,0.9,0.5,0.9,1.8,12.9,NaN,Richmond
4,A.J. Staton-McCray,14.0,SR,G,6-5,190.0,"Charlotte, NC",West Oaks Academy,NaN,"12.0 Pts, 3.3 Reb, 1.4 Ast",...,2.6,3.3,1.4,1.6,0.6,1.3,2.1,12.0,NaN,Seton Hall


In [9]:
df.columns.tolist()

['Player',
 '#',
 'Class',
 'Pos_x',
 'Height',
 'Weight',
 'Hometown',
 'High School',
 'RSCI Top 100',
 'Summary',
 'School_x',
 'Rk',
 'Pos_y',
 'G',
 'GS',
 'MP',
 'FG',
 'FGA',
 'FG%',
 '3P',
 '3PA',
 '3P%',
 '2P',
 '2PA',
 '2P%',
 'eFG%',
 'FT',
 'FTA',
 'FT%',
 'ORB',
 'DRB',
 'TRB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 'Awards',
 'School_y']

In [10]:
def get_role_ratings(df):
    """
    df: A pandas DataFrame containing standard CBB box score stats.
    Required columns: PTS, TRB, ORB, AST, STL, BLK, 3PA, 3P%, FGA, FG%, FTA, TOV, MP
    """
    
    # 1. Normalize the stats (0 to 1 scale)
    # This ensures an '8.0 AST' is weighted appropriately against a '15.0 PTS'
    stats_to_scale = ['PTS', 'TRB', 'ORB', 'AST', 'STL', 'BLK', '3PA', '3P%', 'FGA', 'FG%', 'FTA', 'MP']
    norm_df = df.copy()
    
    for stat in stats_to_scale:
        min_val = df[stat].min()
        max_val = df[stat].max()
        norm_df[stat] = (df[stat] - min_val) / (max_val - min_val)

    # 2. Create Helper metrics
    norm_df['AST_TOV'] = norm_df['AST'] / (df['TOV'] + 1) # Use raw TOV to avoid double-normalizing
    norm_df['3P_Reliance'] = df['3PA'] / (df['FGA'] + 0.1)
    norm_df['Stocks'] = (norm_df['STL'] + norm_df['BLK']) / 2
    
    # 3. Calculate Role Ratings (0.0 to 1.0)
    # We use (1 - norm_df['stat']) for inverse stats (like low FGA or low PTS)
    ratings = pd.DataFrame(index=df.index)
    
    ratings['Floor General'] = (norm_df['AST'] * 0.4) + (norm_df['AST_TOV'] * 0.4) + (norm_df['MP'] * 0.2)
    
    ratings['Pure Sharpshooter'] = (norm_df['3PA'] * 0.4) + (norm_df['3P_Reliance'] * 0.3) + (norm_df['3P%'] * 0.3)
    
    ratings['Slashing Scorer'] = (norm_df['PTS'] * 0.4) + (norm_df['FTA'] * 0.4) + ((1 - norm_df['3P_Reliance']) * 0.2)
    
    ratings['3-and-D Wing'] = (norm_df['Stocks'] * 0.4) + (norm_df['3PA'] * 0.3) + (norm_df['3P%'] * 0.3)
    
    ratings['Traditional Big'] = (norm_df['TRB'] * 0.4) + (norm_df['BLK'] * 0.3) + (norm_df['FG%'] * 0.3)
    
    ratings['Stretch Big'] = (norm_df['3PA'] * 0.4) + (norm_df['TRB'] * 0.3) + (norm_df['3P%'] * 0.3)
    
    ratings['Glass Cleaner'] = (norm_df['ORB'] * 0.5) + (norm_df['TRB'] * 0.3) + ((1 - norm_df['FGA']) * 0.2)
    
    ratings['Connective Wing'] = (norm_df['AST'] * 0.3) + (norm_df['TRB'] * 0.3) + (norm_df['PTS'] * 0.4)
    
    # The Glue Guy: High impact, low usage/scoring
    ratings['Glue Guy'] = (norm_df['Stocks'] * 0.3) + (norm_df['ORB'] * 0.3) + (norm_df['AST'] * 0.2) + ((1 - norm_df['PTS']) * 0.2)

    # 4. Scale to 0-100 for a "Madden Style" rating
    return (ratings * 100).round(1)

# Usage:
# role_results = get_role_ratings(my_scraped_dataframe)

In [17]:
df_role = get_role_ratings(df)
df_role.head()

,Floor General,Pure Sharpshooter,Slashing Scorer,3-and-D Wing,Traditional Big,Stretch Big,Glass Cleaner,Connective Wing,Glue Guy
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11.6,29.1,27.2,18.4,19.3,20.6,21.9,15.8,22.6
2,2.9,NaN,22.1,NaN,30.7,NaN,21.3,2.3,21.0
3,5.8,25.8,25.1,17.8,18.4,17.1,20.3,8.7,21.4
4,6.3,29.0,19.3,18.8,18.3,17.6,21.4,9.3,23.5


In [16]:
df_player_role = pd.concat([df_bio, df_role], axis = 1)
df_player_role.head()

,Player,#,Class,Pos,Height,Weight,Hometown,High School,RSCI Top 100,Summary,School,Floor General,Pure Sharpshooter,Slashing Scorer,3-and-D Wing,Traditional Big,Stretch Big,Glass Cleaner,Connective Wing,Glue Guy
0,Bradyn Hubbard,15.0,SR,F,6-7,225.0,"Tulsa, OK",Tulsa Memorial (OK),NaN,"16.1 Pts, 4.8 Reb, 1.8 Ast",Abilene Christian,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Rich Smith,4.0,SR,G,6-4,175.0,"Bronx, NY",Monsignor Scanlan (NY),NaN,"9.6 Pts, 3.9 Reb, 4.7 Ast",Abilene Christian,11.6,29.1,27.2,18.4,19.3,20.6,21.9,15.8,22.6
2,Chilaydrien Newton,5.0,SO,G,6-3,175.0,"Simsboro, LA",Simsboro (LA); Wasatch Academy (UT),NaN,"9.4 Pts, 2.2 Reb, 0.9 Ast",Abilene Christian,2.9,NaN,22.1,NaN,30.7,NaN,21.3,2.3,21.0
3,Zy Wright,0.0,SR,G,6-5,NaN,"Lincolnton, GA",Grovetown (GA); Aquinas (GA),NaN,"6.2 Pts, 2.4 Reb, 0.7 Ast",Abilene Christian,5.8,25.8,25.1,17.8,18.4,17.1,20.3,8.7,21.4
4,Yaniel Rivera,2.0,JR,G,6-4,175.0,"Bayamon, Puerto Rico",Dade Christian (FL),NaN,"7.0 Pts, 1.5 Reb, 2.5 Ast",Abilene Christian,6.3,29.0,19.3,18.8,18.3,17.6,21.4,9.3,23.5


In [21]:
columns_to_drop = ["#", "Class", "Pos", "Height", "Weight", "Hometown", "High School", "RSCI Top 100", "Summary"]

# Drop columns and assign the result to a new DataFrame
df_player_role = df_player_role.drop(columns=columns_to_drop)


In [23]:
df_player_role[df_player_role["School"] == "Illinois"]

,Player,School,Floor General,Pure Sharpshooter,Slashing Scorer,3-and-D Wing,Traditional Big,Stretch Big,Glass Cleaner,Connective Wing,Glue Guy
1838,Keaton Wagler,Illinois,1.9,NaN,20.5,NaN,9.2,NaN,23.4,1.9,22.1
1839,David Mirkovic,Illinois,3.5,32.2,9.2,14.4,15.9,14.9,22.0,4.7,22.2
1840,Andrej Stojakovic,Illinois,11.6,27.3,33.4,18.3,17.5,18.6,19.6,16.4,22.1
1841,Kylan Boswell,Illinois,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1842,Tomislav Ivisic,Illinois,3.8,4.0,20.4,0.5,14.4,1.7,24.1,3.9,22.5
1843,Zvonimir Ivisic,Illinois,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1844,Ben Humrichous,Illinois,2.2,11.7,19.7,8.6,14.7,9.1,23.1,2.7,22.0
1845,Jake Davis,Illinois,0.2,NaN,20.0,NaN,NaN,NaN,20.0,0.0,20.0
1846,Mihailo Petrovic,Illinois,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1847,Brandon Lee,Illinois,7.1,21.5,19.4,13.0,21.1,14.4,26.4,9.3,24.5


In [33]:

import sqlite3
import random
import tkinter as tk
from tkinter import ttk

DB_NAME = "basketball_sim.db"

# ==========================================
# 1. DATABASE SETUP & DATA GENERATION
# ==========================================
def setup_database():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.executescript('''
        DROP TABLE IF EXISTS teams;
        DROP TABLE IF EXISTS players;
        DROP TABLE IF EXISTS games;
        DROP TABLE IF EXISTS box_scores;
        
        CREATE TABLE teams (id INTEGER PRIMARY KEY, name TEXT);
        CREATE TABLE players (
            id INTEGER PRIMARY KEY, team_id INTEGER, name TEXT,
            inside_shot INTEGER, outside_shot INTEGER, defense INTEGER
        );
        CREATE TABLE games (
            id INTEGER PRIMARY KEY, home_team_id INTEGER, away_team_id INTEGER,
            home_score INTEGER, away_score INTEGER
        );
        CREATE TABLE box_scores (
            id INTEGER PRIMARY KEY, game_id INTEGER, player_id INTEGER,
            team_id INTEGER, points INTEGER
        );
    ''')
    conn.commit()
    return conn

def generate_league(conn):
    c = conn.cursor()
    teams = ["Wildcats", "Spartans", "Bulldogs", "Blue Devils"]
    first_names = ["Marcus", "Jalen", "De'Aaron", "Tyler", "Zion", "Caleb", "Isaiah"]
    last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Davis", "Miller"]

    for team_name in teams:
        c.execute("INSERT INTO teams (name) VALUES (?)", (team_name,))
        team_id = c.lastrowid
        for _ in range(5):
            name = f"{random.choice(first_names)} {random.choice(last_names)}"
            inside = random.randint(50, 99)
            outside = random.randint(50, 99)
            defense = random.randint(50, 99)
            c.execute("""INSERT INTO players (team_id, name, inside_shot, outside_shot, defense) 
                         VALUES (?, ?, ?, ?, ?)""", (team_id, name, inside, outside, defense))
    conn.commit()

# ==========================================
# 2. SIMULATION ENGINE
# ==========================================
def simulate_games(conn):
    c = conn.cursor()
    c.execute("SELECT id FROM teams")
    team_ids = [row[0] for row in c.fetchall()]
    
    # Play each team twice (Home and Away)
    for home in team_ids:
        for away in team_ids:
            if home != away:
                play_game(conn, home, away)

def play_game(conn, home_id, away_id):
    c = conn.cursor()
    c.execute("SELECT id, inside_shot, outside_shot, defense FROM players WHERE team_id=?", (home_id,))
    home_roster = c.fetchall()
    c.execute("SELECT id, inside_shot, outside_shot, defense FROM players WHERE team_id=?", (away_id,))
    away_roster = c.fetchall()
    
    player_points = {p[0]: 0 for p in home_roster + away_roster}
    home_score, away_score = 0, 0
    
    for _ in range(100):
        shooter = random.choice(home_roster)
        defender = random.choice(away_roster)
        pts = resolve_possession(shooter, defender)
        home_score += pts
        player_points[shooter[0]] += pts
        
        shooter = random.choice(away_roster)
        defender = random.choice(home_roster)
        pts = resolve_possession(shooter, defender)
        away_score += pts
        player_points[shooter[0]] += pts

    c.execute("INSERT INTO games (home_team_id, away_team_id, home_score, away_score) VALUES (?, ?, ?, ?)",
              (home_id, away_id, home_score, away_score))
    game_id = c.lastrowid
    
    for p_id, pts in player_points.items():
        t_id = home_id if p_id in [p[0] for p in home_roster] else away_id
        c.execute("INSERT INTO box_scores (game_id, player_id, team_id, points) VALUES (?, ?, ?, ?)",
                  (game_id, p_id, t_id, pts))
    conn.commit()

def resolve_possession(shooter, defender):
    _, s_inside, s_outside, _ = shooter
    _, _, _, d_defense = defender
    
    if random.choice([True, False]):
        chance = 35 + ((s_outside - d_defense) * 0.5) 
        if random.randint(1, 100) < chance: return 3
    else:
        chance = 45 + ((s_inside - d_defense) * 0.5)
        if random.randint(1, 100) < chance: return 2
    return 0

# ==========================================
# 3. TKINTER GUI
# ==========================================
class BasketballSimGUI:
    def __init__(self, root, conn):
        self.root = root
        self.conn = conn
        self.root.title("Text-Based Hoops Sim")
        self.root.geometry("800x500")
        
        # Create Notebook (Tabbed Interface)
        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True, padx=10, pady=10)
        
        # Tab 1: Games & Box Scores
        self.games_tab = tk.Frame(self.notebook)
        self.notebook.add(self.games_tab, text="Schedule & Box Scores")
        
        # Tab 2: Standings
        self.standings_tab = tk.Frame(self.notebook)
        self.notebook.add(self.standings_tab, text="League Standings")
        
        self.setup_games_tab()
        self.setup_standings_tab()

    def setup_games_tab(self):
        left_frame = tk.Frame(self.games_tab, width=250)
        left_frame.pack(side="left", fill="y", padx=10, pady=10)
        right_frame = tk.Frame(self.games_tab)
        right_frame.pack(side="right", fill="both", expand=True, padx=10, pady=10)
        
        tk.Label(left_frame, text="Games Schedule", font=("Arial", 12, "bold")).pack()
        self.games_listbox = tk.Listbox(left_frame, width=30, height=25)
        self.games_listbox.pack(pady=5)
        self.games_listbox.bind('<<ListboxSelect>>', self.show_box_score)
        
        tk.Label(right_frame, text="Box Score", font=("Arial", 12, "bold")).pack()
        cols = ("Player", "Team", "Points")
        self.box_tree = ttk.Treeview(right_frame, columns=cols, show="headings")
        for col in cols:
            self.box_tree.heading(col, text=col)
            self.box_tree.column(col, anchor="center")
        self.box_tree.pack(fill="both", expand=True, pady=5)
        
        self.load_games()

    def setup_standings_tab(self):
        tk.Label(self.standings_tab, text="Current Standings", font=("Arial", 14, "bold")).pack(pady=10)
        
        cols = ("Team", "Wins", "Losses", "Win %")
        self.standings_tree = ttk.Treeview(self.standings_tab, columns=cols, show="headings", height=10)
        
        for col in cols:
            self.standings_tree.heading(col, text=col)
            self.standings_tree.column(col, anchor="center", width=150)
            
        self.standings_tree.pack(pady=10)
        self.load_standings()

    def load_games(self):
        c = self.conn.cursor()
        query = """
            SELECT g.id, t1.name, g.home_score, t2.name, g.away_score 
            FROM games g
            JOIN teams t1 ON g.home_team_id = t1.id
            JOIN teams t2 ON g.away_team_id = t2.id
        """
        self.games_data = {}
        for index, row in enumerate(c.execute(query).fetchall()):
            game_id, home, h_score, away, a_score = row
            display_text = f"{away} {a_score} @ {home} {h_score}"
            self.games_listbox.insert(tk.END, display_text)
            self.games_data[index] = game_id

    def show_box_score(self, event):
        selection = self.games_listbox.curselection()
        if not selection: return
        game_id = self.games_data[selection[0]]
        
        for row in self.box_tree.get_children():
            self.box_tree.delete(row)
            
        c = self.conn.cursor()
        query = """
            SELECT p.name, t.name, b.points 
            FROM box_scores b
            JOIN players p ON b.player_id = p.id
            JOIN teams t ON b.team_id = t.id
            WHERE b.game_id = ?
            ORDER BY t.name, b.points DESC
        """
        for row in c.execute(query, (game_id,)).fetchall():
            self.box_tree.insert("", tk.END, values=row)

    def load_standings(self):
        c = self.conn.cursor()
        # SQL query to calculate Wins, Losses, and Win Percentage dynamically
        query = """
            SELECT 
                t.name,
                SUM(CASE WHEN (g.home_team_id = t.id AND g.home_score > g.away_score) 
                           OR (g.away_team_id = t.id AND g.away_score > g.home_score) THEN 1 ELSE 0 END) as wins,
                SUM(CASE WHEN (g.home_team_id = t.id AND g.home_score < g.away_score) 
                           OR (g.away_team_id = t.id AND g.away_score < g.home_score) THEN 1 ELSE 0 END) as losses
            FROM teams t
            LEFT JOIN games g ON t.id = g.home_team_id OR t.id = g.away_team_id
            GROUP BY t.id
            ORDER BY wins DESC, losses ASC
        """
        
        for row in c.execute(query).fetchall():
            team, wins, losses = row
            # Calculate win percentage, handling division by zero just in case
            total_games = wins + losses
            win_pct = wins / total_games if total_games > 0 else 0.000
            
            # Format as standard basketball win percentage (e.g., .750)
            formatted_win_pct = f"{win_pct:.3f}".lstrip('0')
            if formatted_win_pct == '.000' and wins == 0:
                 formatted_win_pct = '.000'
            elif win_pct == 1.0:
                 formatted_win_pct = '1.000'
            
            self.standings_tree.insert("", tk.END, values=(team, wins, losses, formatted_win_pct))

# ==========================================
# EXECUTION SCRIPT
# ==========================================
if __name__ == "__main__":
    connection = setup_database()
    generate_league(connection)
    simulate_games(connection)
    
    root = tk.Tk()
    app = BasketballSimGUI(root, connection)
    root.mainloop()